# 19 · Multi-Query：一个问题，多路召回

> 一个“说法”可能错过很多相关内容。让 LLM 从不同角度改写出多个查询，各自召回再融合，召回率明显提升。

**本文件覆盖知识点**：Multi-Query / Parallel Retrieval / 多查询生成

```text
"Redis为什么快?"
  → "Redis性能为什么高？"
  → "Redis为什么采用单线程？"
  → "Redis IO模型是什么？"
  → "Redis为什么比MySQL快？"      ← 各自检索，合并去重
```

In [ ]:
# .env 配置
from dotenv import load_dotenv; load_dotenv()
import os, json
from dashscope import Generation
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def generate_queries(query, n=3):
    """把一个查询扩展成 n 个不同角度的检索查询，输出 JSON 数组"""
    prompt = (f'把下面问题改写成 {n} 个意思相关但侧重不同的检索查询，直接输出 JSON 字符串数组，不解释。\n问题: {query}')
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':prompt}],
                        api_key=API_KEY, result_format='message')
    raw = r.output.choices[0].message.content.strip().strip('`')
    if raw.startswith('json'): raw = raw[4:].strip()
    try:
        return [query] + json.loads(raw)   # 原查询也保留
    except Exception:
        return [query]

if API_KEY and '你的' not in API_KEY:
    for q in generate_queries('星云客服机器人怎么收费', 3):
        print('•', q)

## 检索与融合

Multi-Query 的检索与融合和普通检索一致：
```text
q1 ──→ 检索 ──┐
q2 ──→ 检索 ──┼─→ 合并(去重/按名次 RRF) ──→ 候选池
q3 ──→ 检索 ──┘
```

- **合并手段**：简单拼接去重，或按 17 课 **RRF** 融合多个榜单；
- **并行执行**：多个查询相互独立，应并发请求以省时（`concurrent.futures` / 异步）；
- **代价**：多 n 次检索 → 延迟与成本上升，注意控制 n 与候选池大小。

In [ ]:
# 融合演示：几个榜单用 17 课的 RRF 合并成一个最终榜单
def rrf_fuse(rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] = score.get(doc, 0) + 1.0 / (k + rank + 1)
    return sorted(score, key=score.get, reverse=True)

rankings = [
    ['doc5','doc1','doc9'],   # 查询1召回的
    ['doc9','doc3','doc1'],   # 查询2召回的
    ['doc1','doc5'],          # 查询3召回的
]
print('多路召回融合排序:', rrf_fuse(rankings))
print('=> doc1 在 3 个榜单都靠前 → 排第一，比单个查询更稳。')

## 小结

- Multi-Query = **多角度改写 + 并行召回 + 融合去重**；
- 融合建议 RRF，召回建议并发；
- 与 RAG-Fusion（下一课）是“同思路+结构化组装”。

如果连“扩展出的查询”本身也想优化（同义词加权、按查询加权融合），就到了 RAG-Fusion 与 Query Expansion。